# Task 2B — Map MAR Points to Pixels + Create Binary Dataset

Converted from `Task_2B_MAR_PixelValues.py` on 2025-12-29 08:46:04.

Run cells **top-to-bottom**. This notebook is broken into small steps with prints/previews so you can **see outputs for every step**.

## 1) Library imports

In [2]:
import os
import pandas as pd
import geopandas as gpd
import rasterio

print("Imports OK.")
print("pandas:", pd.__version__)
print("geopandas:", gpd.__version__)
print("rasterio:", rasterio.__version__)


Imports OK.
pandas: 2.3.3
geopandas: 1.1.2
rasterio: 1.4.4


## 2) User inputs (Edit this cell) 
Update your input/output paths here.

In [3]:
# Existing pixel table (from Task 2A output)
PIXEL_CSV = r"E:\VUB\Final\PixelDataFrames\pixels_2014_2024_all_inside.csv"

# Managed Aquifer Recharge (MAR) shapefile (already clipped to study area)
MAR_SHP   = r"E:\VUB\Final\Boundaries\MAR_EU.shp"
MAR_FIELD = "main_mar_t"  # attribute field containing MAR type/category

# Reference raster for point → pixel_id conversion (must match the pixel_id logic used in Task 2A)
REF_RASTER = r"E:\VUB\Final\AET_Clipped\AET_2014_clipped.tif"

# Outputs
OUT_MAR_SAMPLES = r"E:\VUB\Final\PixelDataFrames\mar_samples_pixels.csv"
OUT_BINARY_CSV  = r"E:\VUB\Final\PixelDataFrames\mar_binary_dataset_all_years.csv"

# If MAR locations are static across all years
STATIC_MAR = True
MAR_YEAR = 2014   # used only if STATIC_MAR = False

print("PIXEL_CSV:", PIXEL_CSV)
print("MAR_SHP:", MAR_SHP)
print("REF_RASTER:", REF_RASTER)
print("OUT_MAR_SAMPLES:", OUT_MAR_SAMPLES)
print("OUT_BINARY_CSV:", OUT_BINARY_CSV)
print("STATIC_MAR:", STATIC_MAR, "| MAR_YEAR:", MAR_YEAR)


PIXEL_CSV: E:\VUB\Final\PixelDataFrames\pixels_2014_2024_all_inside.csv
MAR_SHP: E:\VUB\Final\Boundaries\MAR_EU.shp
REF_RASTER: E:\VUB\Final\AET_Clipped\AET_2014_clipped.tif
OUT_MAR_SAMPLES: E:\VUB\Final\PixelDataFrames\mar_samples_pixels.csv
OUT_BINARY_CSV: E:\VUB\Final\PixelDataFrames\mar_binary_dataset_all_years.csv
STATIC_MAR: True | MAR_YEAR: 2014


## 3) Pre checks
Ensures files exist and output folders are writable.

In [4]:
def ensure_dir(path):
    os.makedirs(os.path.dirname(path), exist_ok=True)

missing = False
for p, name in [(PIXEL_CSV,"PIXEL_CSV"), (MAR_SHP,"MAR_SHP"), (REF_RASTER,"REF_RASTER")]:
    if not os.path.exists(p):
        print("❌ Missing:", name, "->", p)
        missing = True
    else:
        print("✅ Found:", name)

ensure_dir(OUT_MAR_SAMPLES)
ensure_dir(OUT_BINARY_CSV)
print("✅ Output dirs ready")

if missing:
    raise FileNotFoundError("Fix missing paths in Config cell and rerun.")


✅ Found: PIXEL_CSV
✅ Found: MAR_SHP
✅ Found: REF_RASTER
✅ Output dirs ready


## 4) Helpers (category normalization + stable mode)

In [5]:
def norm(s):
    """Normalize category string (case-insensitive, trim spaces)."""
    return " ".join(str(s).strip().lower().split())

def mode_category(s: pd.Series):
    """Stable mode (alphabetical tie-break)."""
    vc = s.value_counts()
    top = vc[vc == vc.max()].index.tolist()
    return sorted(top)[0]

print("Helpers loaded.")


Helpers loaded.


## 5) Load inputs + quick preview (sanity check)

In [6]:
df_pixels = pd.read_csv(PIXEL_CSV)
gmar = gpd.read_file(MAR_SHP)

print("Pixel table rows:", len(df_pixels))
print("Pixel table cols:", list(df_pixels.columns)[:20], "..." if len(df_pixels.columns) > 20 else "")
print("MAR points:", len(gmar))
print("MAR columns:", list(gmar.columns))

display(df_pixels.head(5))
display(gmar.head(5))


Pixel table rows: 614953
Pixel table cols: ['year', 'pixel_id', 'row', 'col', 'lon', 'lat', 'AET', 'LULC', 'P', 'RZSM', 'TEMP', 'SOIL'] 
MAR points: 279
MAR columns: ['fid', 'id', 'site_name', 'continent', 'country', 'latitude', 'longitude', 'year_opera', 'year_shut_', 'main_mar_t', 'specific_m', 'link_to_do', 'influent_s', 'final_use', 'main_objec', 'reference_', 'geometry']


,year,pixel_id,row,col,lon,lat,AET,LULC,P,RZSM,TEMP,SOIL
0,2014,382,0,382,27.55,70.05,204.13990,NaN,238.71922,0.270068,-0.036160,4.0
1,2014,383,0,383,27.65,70.05,206.53160,30.0,233.09563,0.270511,-0.002954,4.0
2,2014,384,0,384,27.75,70.05,201.74350,30.0,229.78244,0.293638,0.038191,4.0
3,2014,385,0,385,27.85,70.05,213.14160,30.0,229.21902,0.272055,0.080452,4.0
4,2014,386,0,386,27.95,70.05,228.95428,30.0,230.52446,0.347064,0.099918,4.0


,fid,id,site_name,continent,country,latitude,longitude,year_opera,year_shut_,main_mar_t,specific_m,link_to_do,influent_s,final_use,main_objec,reference_,geometry
0,18569,486.0,Lake Baltezers,Europe,Latvia,56.95471,24.10538,-9999.0,-9999.0,Induced Bank Filtration,Induced Bank Filtration,"<a href=""https://inowas.com/mar-methods/induce...",River water,Domestic,Maximize Natural Storage,"Grischek, T., Schoenheinz, D., Worch, E., Hisc...",POINT (24.10538 56.95471)
1,18557,474.0,Borsodszirak,Europe,Hungary,48.26100,20.76800,-9999.0,-9999.0,Spreading Methods,Infiltration Ponds and Basins,"<a href=""https://inowas.com/mar-methods/infilt...",River water,no data,no data,"<a href=""https://www.hydrology.nl/images/docs/...",POINT (20.768 48.261)
2,18648,567.0,Gheraiesti,Europe,Romania,46.59933,26.90449,1961.0,-9999.0,Induced Bank Filtration,Induced Bank Filtration,"<a href=""https://inowas.com/mar-methods/induce...",River water,Domestic,Water Quality Management,"Grischek, T., Schoenheinz, D., Worch, E., Hisc...",POINT (26.90449 46.59933)
3,18649,568.0,Cluj,Europe,Romania,46.74750,23.49080,1935.0,-9999.0,Induced Bank Filtration,Induced Bank Filtration,"<a href=""https://inowas.com/mar-methods/induce...",River water,Domestic,Water Quality Management,"Rojanschi, V., Mlenanek, L., Stacilescu, M., 2...",POINT (23.4908 46.7475)
4,18650,569.0,Iasi,Europe,Romania,47.15690,27.59030,1911.0,-9999.0,Induced Bank Filtration,Induced Bank Filtration,"<a href=""https://inowas.com/mar-methods/induce...",River water,Domestic,Water Quality Management,"Rojanschi, V., Mlenanek, L., Stacilescu, M., 2...",POINT (27.5903 47.1569)


## 6) Step A — Sample MAR shapefile → pixels (creates `OUT_MAR_SAMPLES`)

In [7]:
print("\n=== STEP A: MAR SAMPLING ===")

if MAR_FIELD not in gmar.columns:
    raise ValueError(f"Field '{MAR_FIELD}' not found in MAR shapefile")

valid_pixels = set(df_pixels["pixel_id"].unique())
print("Unique pixels in PIXEL_CSV:", len(valid_pixels))

# ---- Normalize MAR categories ----
CATEGORIES_STD = [
    "In-Channel Modification",
    "Induced Bank Filtration",
    "Rainwater and Run-off Harvesting",
    "Spreading Methods",
    "Well, Shaft and Borehole Recharge",
]

norm_to_std = {norm(c): c for c in CATEGORIES_STD}
cat_to_code = {c: i for i, c in enumerate(CATEGORIES_STD)}

gmar2 = gmar.dropna(subset=[MAR_FIELD]).copy()
gmar2["_cat_norm"] = gmar2[MAR_FIELD].astype(str).apply(norm)
gmar2 = gmar2[gmar2["_cat_norm"].isin(norm_to_std)].copy()
gmar2["MAR_label"] = gmar2["_cat_norm"].map(norm_to_std)
gmar2.drop(columns="_cat_norm", inplace=True)

print("Total MAR points:", len(gmar))
print("After category filtering:", len(gmar2))

# ---- Convert MAR points to pixel_id ----
with rasterio.open(REF_RASTER) as ref:
    if gmar2.crs != ref.crs:
        gmar2 = gmar2.to_crs(ref.crs)

    xs = gmar2.geometry.x.to_numpy()
    ys = gmar2.geometry.y.to_numpy()
    rows, cols = rasterio.transform.rowcol(ref.transform, xs, ys)
    W = ref.width

gmar2["row"] = rows
gmar2["col"] = cols
gmar2["pixel_id"] = gmar2["row"].astype("int64") * W + gmar2["col"].astype("int64")

# ---- Guard: keep only pixels inside the pixel table domain ----
before = len(gmar2)
gmar2 = gmar2[gmar2["pixel_id"].isin(valid_pixels)].copy()
print(f"After pixel guard: {len(gmar2)} (removed {before - len(gmar2)})")

# ---- One MAR label per pixel (mode) ----
gmar_mode = (
    gmar2.groupby("pixel_id")["MAR_label"]
        .apply(mode_category)
        .reset_index()
)
gmar_mode["MAR_code"] = gmar_mode["MAR_label"].map(cat_to_code).astype(int)

# ---- Merge with pixel table ----
if STATIC_MAR:
    samples = df_pixels.merge(gmar_mode, on="pixel_id", how="inner")
else:
    df_year = df_pixels[df_pixels["year"] == MAR_YEAR].copy()
    samples = df_year.merge(gmar_mode, on="pixel_id", how="inner")

samples["pixel_year_id"] = samples["pixel_id"].astype(str) + "_" + samples["year"].astype(str)

samples.to_csv(OUT_MAR_SAMPLES, index=False)

print("Unique MAR pixels:", len(gmar_mode))
print("MAR samples saved:", OUT_MAR_SAMPLES)
display(samples.head(10))



=== STEP A: MAR SAMPLING ===
Unique pixels in PIXEL_CSV: 55929
Total MAR points: 279
After category filtering: 279
After pixel guard: 254 (removed 25)
Unique MAR pixels: 219
MAR samples saved: E:\VUB\Final\PixelDataFrames\mar_samples_pixels.csv


,year,pixel_id,row,col,lon,lat,AET,LULC,P,RZSM,TEMP,SOIL,MAR_label,MAR_code,pixel_year_id
0,2014,21563,47,272,16.55,65.35,172.00357,111.0,728.16570,NaN,2.225394,4.0,Spreading Methods,3,21563_2014
1,2014,32999,72,383,27.65,62.85,565.99664,111.0,716.23865,0.310972,5.217053,4.0,Induced Bank Filtration,1,32999_2014
2,2014,34304,75,329,22.25,62.55,499.93250,111.0,625.83234,0.304080,5.714527,4.0,Induced Bank Filtration,1,34304_2014
3,2014,35698,78,364,25.75,62.25,485.78363,111.0,650.00620,0.318120,5.517731,4.0,Spreading Methods,3,35698_2014
4,2014,38431,84,379,27.25,61.65,477.28930,111.0,698.70950,0.315008,5.782207,4.0,Spreading Methods,3,38431_2014
5,2014,39300,86,342,23.55,61.45,544.11880,80.0,665.37850,0.377936,6.207617,4.0,Induced Bank Filtration,1,39300_2014
6,2014,39305,86,347,24.05,61.45,542.01135,80.0,662.54160,0.378817,6.070190,3.0,Induced Bank Filtration,1,39305_2014
7,2014,39759,87,348,24.15,61.35,601.61237,80.0,655.03986,0.377151,6.127059,NaN,Induced Bank Filtration,1,39759_2014
8,2014,40236,88,372,26.55,61.25,560.33820,111.0,687.08450,0.325879,6.105983,4.0,Spreading Methods,3,40236_2014
9,2014,41158,90,388,28.15,61.05,522.67780,80.0,722.45795,0.356473,6.137761,4.0,Induced Bank Filtration,1,41158_2014


## 7) Step B — Create binary dataset for ML (creates `OUT_BINARY_CSV`)

In [8]:
print("\n=== STEP B: BINARY DATASET ===")

df_all = pd.read_csv(PIXEL_CSV)
df_mar = pd.read_csv(OUT_MAR_SAMPLES)

print("Total pixel records:", len(df_all))
print("Total MAR sample records:", len(df_mar))

# Default: unsuitable
df_all["MAR_suitable"] = 0
mar_pixels = set(df_mar["pixel_id"].unique())
df_all.loc[df_all["pixel_id"].isin(mar_pixels), "MAR_suitable"] = 1

# Auto-detect feature columns
EXCLUDE = {
    "year", "pixel_id", "row", "col", "lon", "lat",
    "MAR_label", "MAR_code", "pixel_year_id", "MAR_suitable"
}
feature_cols = [c for c in df_all.columns if c not in EXCLUDE]

final_cols = ["year", "pixel_id", "lon", "lat"] + feature_cols + ["MAR_suitable"]
df_final = df_all[final_cols].dropna(subset=feature_cols, how="any")

df_final.to_csv(OUT_BINARY_CSV, index=False)

print("Binary dataset saved:", OUT_BINARY_CSV)
print("Class distribution:")
print(df_final["MAR_suitable"].value_counts())
display(df_final.head(10))



=== STEP B: BINARY DATASET ===
Total pixel records: 614953
Total MAR sample records: 2409
Binary dataset saved: E:\VUB\Final\PixelDataFrames\mar_binary_dataset_all_years.csv
Class distribution:
MAR_suitable
0    511566
1      1848
Name: count, dtype: int64


,year,pixel_id,lon,lat,AET,LULC,P,RZSM,TEMP,SOIL,MAR_suitable
1,2014,383,27.65,70.05,206.53160,30.0,233.09563,0.270511,-0.002954,4.0,0
2,2014,384,27.75,70.05,201.74350,30.0,229.78244,0.293638,0.038191,4.0,0
3,2014,385,27.85,70.05,213.14160,30.0,229.21902,0.272055,0.080452,4.0,0
4,2014,386,27.95,70.05,228.95428,30.0,230.52446,0.347064,0.099918,4.0,0
5,2014,826,26.65,69.95,247.35196,30.0,262.63400,0.258783,-0.309810,4.0,0
7,2014,828,26.85,69.95,242.82750,30.0,256.99430,0.272365,-0.171541,4.0,0
8,2014,833,27.35,69.95,218.71545,30.0,485.81534,0.258491,-0.224810,4.0,0
9,2014,834,27.45,69.95,201.59355,30.0,537.55870,0.289886,-0.236375,4.0,0
10,2014,835,27.55,69.95,201.30743,30.0,527.17584,0.325407,-0.205651,4.0,0
11,2014,836,27.65,69.95,201.70766,30.0,517.31040,0.337169,-0.142084,4.0,0


## 8) Optional: quick checks (duplicates + missingness)

In [9]:
# Check if any pixel_id appears multiple times in mar samples (should be 1 per pixel_id)
if os.path.exists(OUT_MAR_SAMPLES):
    s = pd.read_csv(OUT_MAR_SAMPLES)
    dup = s["pixel_id"].duplicated().sum()
    print("Duplicate pixel_id in mar_samples_pixels.csv:", dup)

# Missingness summary in final dataset
if os.path.exists(OUT_BINARY_CSV):
    d = pd.read_csv(OUT_BINARY_CSV, nrows=500000)  # sample
    miss = d.isna().mean().sort_values(ascending=False).head(15)
    print("\nTop missingness (sample):")
    print(miss)


Duplicate pixel_id in mar_samples_pixels.csv: 2190

Top missingness (sample):
year            0.0
pixel_id        0.0
lon             0.0
lat             0.0
AET             0.0
LULC            0.0
P               0.0
RZSM            0.0
TEMP            0.0
SOIL            0.0
MAR_suitable    0.0
dtype: float64
